In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz, freqs, bilinear
from ipywidgets import HTML, Layout
from IPython.display import display

# ============================================================
# BUTTERWORTH IIR DESIGN USING THE BILINEAR TRANSFORMATION
# Complete analytical solution and independent SciPy verification
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.bt-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.bt-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:11px 15px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.bt-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:11px 14px;
    border-radius:0 0 8px 8px;
    font-size:15px;
    line-height:1.55;
    margin-bottom:9px;
}

.bt-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:10px 12px;
    margin-bottom:8px;
    font-size:14.5px;
    line-height:1.50;
}

.bt-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:15.5px;
    margin-bottom:6px;
}

.bt-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.bt-col{
    flex:1;
    min-width:0;
}

.bt-note{
    background:#fff9e8;
    border:1px solid #d9c477;
}

.bt-ok{
    background:#eef7ee;
    border:1px solid #9cc79c;
}

.bt-code{
    text-align:center;
    font-family:Consolas,monospace;
    font-size:14px;
    margin:9px 0;
}

.bt-result-row{
    margin:3px 0;
    white-space:nowrap;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="bt-root">

<div class="bt-header">
Butterworth IIR Filter Design Using the Bilinear Transformation
</div>

<div class="bt-doc">

<b>Design specifications.</b>
Construct a digital Butterworth IIR filter using the bilinear transformation with

<div style="text-align:center;font-size:16px;margin:10px 0;">
<b>
ω<sub>p</sub> = 0.15π,
&nbsp;&nbsp;
ω<sub>s</sub> = 0.35π,
&nbsp;&nbsp;
A<sub>p</sub> = 3 dB,
&nbsp;&nbsp;
A<sub>s</sub> = 20 dB.
</b>
</div>

The complete design is carried out explicitly from the theoretical equations.
The digital edge frequencies are first <b>prewarped</b>, the Butterworth order and
cutoff frequency are determined, the analog prototype is constructed, and the
bilinear substitution

<div style="text-align:center;font-size:16px;margin:9px 0;">
<b>
s =
(2/T<sub>s</sub>)
(1-z<sup>-1</sup>) /
(1+z<sup>-1</sup>)
</b>
</div>

is then applied to obtain the digital transfer function.

For simplicity, and exactly as in the analytical solution, we set

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>T<sub>s</sub> = 1.</b>
</div>

Only after the complete solution has been obtained, the SciPy function

<div class="bt-code">
<b>scipy.signal.bilinear(...)</b>
</div>

is used as an <b>independent software verification</b> of the final coefficients.

</div>

</div>
"""))

# ============================================================
# SPECIFICATIONS
# ============================================================

Ts = 1.0

wp = 0.15*np.pi
ws = 0.35*np.pi

Ap = 3.0
As = 20.0

# ============================================================
# STEP 1 — PREWARP DIGITAL EDGE FREQUENCIES
# ============================================================

Omega_p = (2.0/Ts)*np.tan(wp/2.0)
Omega_s = (2.0/Ts)*np.tan(ws/2.0)

# ============================================================
# STEP 2 — BUTTERWORTH ORDER
# ============================================================

N_real = 0.5*np.log((10**(As/10.0)-1.0)/(10**(Ap/10.0)-1.0))/np.log(Omega_s/Omega_p)

N = int(np.ceil(N_real))

# ============================================================
# STEP 3 — DIGITAL CUTOFF FREQUENCY
# ============================================================

wc = wp/(10**(Ap/10.0)-1.0)**(1.0/(2*N))

# ============================================================
# STEP 4 — PREWARP CUTOFF FREQUENCY
# ============================================================

Omega_c = (2.0/Ts)*np.tan(wc/2.0)

# ============================================================
# STEP 5 — ANALOG BUTTERWORTH POLES
# ============================================================

p1 = -Omega_c
p2 = Omega_c*(-0.5+1j*np.sqrt(3.0)/2.0)
p3 = Omega_c*(-0.5-1j*np.sqrt(3.0)/2.0)

analog_poles = np.array([p1,p2,p3])

# ============================================================
# STEP 6 — ANALOG TRANSFER FUNCTION
# ============================================================

analog_den = np.real_if_close(np.poly(analog_poles)).astype(float)

analog_num = np.array([Omega_c**N],dtype=float)

# ============================================================
# STEP 7 — EXPLICIT BILINEAR SUBSTITUTION
# ============================================================

a1 = analog_den[1]
a2 = analog_den[2]
a3 = analog_den[3]

P_plus_3 = np.array([1.0,3.0,3.0,1.0])

P_s1 = np.convolve(np.array([1.0,-1.0]),np.array([1.0,2.0,1.0]))

P_s2 = np.convolve(np.array([1.0,-2.0,1.0]),np.array([1.0,1.0]))

P_s3 = np.array([1.0,-3.0,3.0,-1.0])

digital_den_raw = 8.0*P_s3+4.0*a1*P_s2+2.0*a2*P_s1+a3*P_plus_3

digital_num_raw = analog_num[0]*P_plus_3

norm = digital_den_raw[0]

digital_den = digital_den_raw/norm

digital_num = digital_num_raw/norm

# ============================================================
# STEP 8 — DIRECT FREQUENCY-RESPONSE EVALUATION
# ============================================================

def evaluate_Hz(b,a,omega):

    q = np.exp(-1j*omega)

    numerator = np.sum(b*q**np.arange(len(b)))

    denominator = np.sum(a*q**np.arange(len(a)))

    return numerator/denominator

H_wp = evaluate_Hz(digital_num,digital_den,wp)

H_ws = evaluate_Hz(digital_num,digital_den,ws)

mag_wp = np.abs(H_wp)

mag_ws = np.abs(H_ws)

db_wp = 20.0*np.log10(mag_wp)

db_ws = 20.0*np.log10(mag_ws)

# ============================================================
# ANALOG FREQUENCY RESPONSE
# ============================================================

Omega_plot = np.logspace(-2,2,4000)

Omega_plot,Ha = freqs(analog_num,analog_den,worN=Omega_plot)

Ha_dB = 20.0*np.log10(np.maximum(np.abs(Ha),1e-12))

# ============================================================
# DIGITAL FREQUENCY RESPONSE
# ============================================================

omega,Hd = freqz(digital_num,digital_den,worN=32768)

Hd_dB = 20.0*np.log10(np.maximum(np.abs(Hd),1e-12))

# ============================================================
# DIGITAL POLES
# ============================================================

digital_poles = np.roots(digital_den)

# ============================================================
# FINAL SOFTWARE VERIFICATION WITH SCIPY
# ============================================================

scipy_num,scipy_den = bilinear(analog_num,analog_den,fs=1.0/Ts)

scipy_num[np.abs(scipy_num)<1e-12] = 0.0

scipy_den[np.abs(scipy_den)<1e-12] = 0.0

numerator_difference = np.max(np.abs(digital_num-scipy_num))

denominator_difference = np.max(np.abs(digital_den-scipy_den))

verification_passed = np.allclose(digital_num,scipy_num,rtol=1e-10,atol=1e-10) and np.allclose(digital_den,scipy_den,rtol=1e-10,atol=1e-10)

# ============================================================
# DISPLAY — PREWARPING AND ORDER
# ============================================================

display(HTML(f"""
<div class="bt-root">

<div class="bt-box">

<div class="bt-title">Step 1 — Prewarping and Butterworth order</div>

<div class="bt-cols">

<div class="bt-col">
Digital passband edge:<br>
<b>ω<sub>p</sub> = {wp:.6f}</b>
<br><br>
Prewarped passband edge:<br>
<b>Ω<sub>p</sub> = {Omega_p:.6f}</b>
</div>

<div class="bt-col">
Digital stopband edge:<br>
<b>ω<sub>s</sub> = {ws:.6f}</b>
<br><br>
Prewarped stopband edge:<br>
<b>Ω<sub>s</sub> = {Omega_s:.6f}</b>
</div>

<div class="bt-col">
Frequency ratio:<br>
<b>Ω<sub>s</sub>/Ω<sub>p</sub> = {Omega_s/Omega_p:.6f}</b>
<br><br>
Calculated order:<br>
<b>N* = {N_real:.6f}</b>
</div>

<div class="bt-col">
Required integer order:<br>
<b>N = {N}</b>
<br><br>
Sampling period:<br>
<b>T<sub>s</sub> = {Ts:.1f}</b>
</div>

</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — CUTOFF FREQUENCY
# ============================================================

display(HTML(f"""
<div class="bt-root">

<div class="bt-box">

<div class="bt-title">Step 2 — Cutoff frequency</div>

The digital cutoff frequency satisfying the passband requirement is

<div style="text-align:center;font-size:15.5px;margin:8px 0;">
<b>
ω<sub>c</sub> = {wc:.6f}.
</b>
</div>

After prewarping,

<div style="text-align:center;font-size:15.5px;margin:8px 0;">
<b>
Ω<sub>c</sub> =
(2/T<sub>s</sub>) tan(ω<sub>c</sub>/2)
= {Omega_c:.6f}.
</b>
</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — ANALOG PROTOTYPE
# ============================================================

display(HTML(f"""
<div class="bt-root">

<div class="bt-box">

<div class="bt-title">Step 3 — Analog Butterworth prototype</div>

<div class="bt-cols">

<div class="bt-col">
Real pole:<br>
<b>
p₁ =
{np.real(p1):.6f}
{np.imag(p1):+.6f}j
</b>
</div>

<div class="bt-col">
Complex pole:<br>
<b>
p₂ =
{np.real(p2):.6f}
{np.imag(p2):+.6f}j
</b>
</div>

<div class="bt-col">
Complex pole:<br>
<b>
p₃ =
{np.real(p3):.6f}
{np.imag(p3):+.6f}j
</b>
</div>

</div>

<br>

The analog transfer function is

<div style="text-align:center;font-size:15px;line-height:1.8;margin:8px 0;">
<b>
H<sub>a</sub>(s) =
{analog_num[0]:.8f}
/
[s³ + {analog_den[1]:.8f}s²
+ {analog_den[2]:.8f}s
+ {analog_den[3]:.8f}].
</b>
</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — BILINEAR SUBSTITUTION
# ============================================================

display(HTML(f"""
<div class="bt-root">

<div class="bt-box bt-note">

<div class="bt-title">Step 4 — Explicit bilinear transformation</div>

Using

<div style="text-align:center;font-size:15.5px;margin:8px 0;">
<b>
s =
2(1-z<sup>-1</sup>) /
(1+z<sup>-1</sup>)
</b>
</div>

for T<sub>s</sub>=1, and multiplying numerator and denominator by
<b>(1+z<sup>-1</sup>)³</b>, the final digital transfer function becomes

<div style="text-align:center;font-size:15px;line-height:1.8;margin:10px 0;">
<b>
H(z) =
({digital_num[0]:.8f}
{digital_num[1]:+.8f}z<sup>-1</sup>
{digital_num[2]:+.8f}z<sup>-2</sup>
{digital_num[3]:+.8f}z<sup>-3</sup>)
/
(1
{digital_den[1]:+.8f}z<sup>-1</sup>
{digital_den[2]:+.8f}z<sup>-2</sup>
{digital_den[3]:+.8f}z<sup>-3</sup>).
</b>
</div>

</div>

</div>
"""))

# ============================================================
# DISPLAY — SPECIFICATION CHECK
# ============================================================

display(HTML(f"""
<div class="bt-root">

<div class="bt-box bt-ok">

<div class="bt-title">Step 5 — Verification of the filter specifications</div>

<div class="bt-cols">

<div class="bt-col">

At ω<sub>p</sub> = 0.15π:

<div class="bt-result-row">
<b>
H(e<sup>jωp</sup>) =
{np.real(H_wp):.6f}
{np.imag(H_wp):+.6f}j
</b>
</div>

<div class="bt-result-row">
<b>|H| = {mag_wp:.6f}</b>
</div>

<div class="bt-result-row">
Magnitude = <b>{db_wp:.6f} dB</b>
</div>

</div>

<div class="bt-col">

At ω<sub>s</sub> = 0.35π:

<div class="bt-result-row">
<b>
H(e<sup>jωs</sup>) =
{np.real(H_ws):.6f}
{np.imag(H_ws):+.6f}j
</b>
</div>

<div class="bt-result-row">
<b>|H| = {mag_ws:.6f}</b>
</div>

<div class="bt-result-row">
Magnitude = <b>{db_ws:.6f} dB</b>
</div>

</div>

</div>

<br>

The passband specification is satisfied essentially at
<b>-3 dB</b>, while the stopband attenuation exceeds the required
<b>20 dB</b>.

</div>

</div>
"""))

# ============================================================
# DISPLAY — SCIPY SOFTWARE VERIFICATION
# ============================================================

display(HTML(f"""
<div class="bt-root">

<div class="bt-box bt-ok">

<div class="bt-title">Final software verification with SciPy</div>

The complete bilinear-transformation design above was obtained explicitly from
the theoretical equations.

<br><br>

Only now do we independently apply

<div class="bt-code">
<b>scipy.signal.bilinear(analog_num, analog_den, fs=1/T<sub>s</sub>)</b>
</div>

<div class="bt-cols">

<div class="bt-col">
<b>Explicit numerator</b><br>
[{digital_num[0]:.10f}, {digital_num[1]:.10f}, {digital_num[2]:.10f}, {digital_num[3]:.10f}]
</div>

<div class="bt-col">
<b>SciPy numerator</b><br>
[{scipy_num[0]:.10f}, {scipy_num[1]:.10f}, {scipy_num[2]:.10f}, {scipy_num[3]:.10f}]
</div>

</div>

<br>

<div class="bt-cols">

<div class="bt-col">
<b>Explicit denominator</b><br>
[{digital_den[0]:.10f}, {digital_den[1]:.10f}, {digital_den[2]:.10f}, {digital_den[3]:.10f}]
</div>

<div class="bt-col">
<b>SciPy denominator</b><br>
[{scipy_den[0]:.10f}, {scipy_den[1]:.10f}, {scipy_den[2]:.10f}, {scipy_den[3]:.10f}]
</div>

</div>

<br>

Maximum numerator difference:
<b>{numerator_difference:.3e}</b>

&nbsp;&nbsp;&nbsp;&nbsp;

Maximum denominator difference:
<b>{denominator_difference:.3e}</b>

<br><br>

<div style="text-align:center;font-size:16px;">
<b>
VERIFICATION:
{"PASSED — the explicit calculation agrees with scipy.signal.bilinear()." if verification_passed else "FAILED — the two calculations do not agree within numerical precision."}
</b>
</div>

</div>

</div>
"""))

# ============================================================
# FIGURE — 2 x 2 GRID
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.6))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. ANALOG POLES
# ============================================================

ax1.axhline(0,color='black',linewidth=0.8)

ax1.axvline(0,color='black',linewidth=0.8)

ax1.axvspan(-0.8,0,alpha=0.05)

ax1.plot(np.real(analog_poles),np.imag(analog_poles),'rx',markersize=8,markeredgewidth=1.8,label='Analog poles')

ax1.set_xlim(-0.7,0.2)

ax1.set_ylim(-0.6,0.6)

ax1.set_title('Analog Butterworth Poles')

ax1.set_xlabel(r'$\Re\{s\}$')

ax1.set_ylabel(r'$\Im\{s\}$')

ax1.grid(True,linestyle=':',alpha=0.25)

ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),frameon=False)

# ============================================================
# 2. DIGITAL POLES
# ============================================================

theta = np.linspace(0,2*np.pi,1000)

ax2.axhline(0,color='black',linewidth=0.8)

ax2.axvline(0,color='black',linewidth=0.8)

ax2.plot(np.cos(theta),np.sin(theta),'--',linewidth=1.1,label='Unit circle')

ax2.plot(np.real(digital_poles),np.imag(digital_poles),'rx',markersize=8,markeredgewidth=1.8,label='Digital poles')

ax2.set_xlim(-1.15,1.15)

ax2.set_ylim(-1.15,1.15)

ax2.set_aspect('equal',adjustable='box')

ax2.set_title('Digital Poles after Bilinear Transformation')

ax2.set_xlabel(r'$\Re\{z\}$')

ax2.set_ylabel(r'$\Im\{z\}$')

ax2.grid(True,linestyle=':',alpha=0.25)

ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 3. ANALOG MAGNITUDE RESPONSE
# ============================================================

ax3.semilogx(Omega_plot,Ha_dB,color='red',linewidth=1.4,label='Analog magnitude response')

ax3.axvline(Omega_p,linestyle='--',linewidth=1.0,label=r'$\Omega_p$')

ax3.axvline(Omega_s,linestyle=':',linewidth=1.0,label=r'$\Omega_s$')

ax3.set_xlim(1e-2,1e2)

ax3.set_ylim(-100,5)

ax3.set_title('Analog Butterworth Magnitude Response')

ax3.set_xlabel(r'Analog frequency $\Omega$')

ax3.set_ylabel('Magnitude (dB)')

ax3.grid(True,which='both',linestyle=':',alpha=0.25)

ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=3,frameon=False)

# ============================================================
# 4. DIGITAL MAGNITUDE RESPONSE
# ============================================================

ax4.plot(omega/np.pi,Hd_dB,color='red',linewidth=1.4,label='Digital magnitude response')

ax4.axvline(0.15,linestyle='--',linewidth=1.0,label=r'$\omega_p=0.15\pi$')

ax4.axvline(0.35,linestyle=':',linewidth=1.0,label=r'$\omega_s=0.35\pi$')

ax4.plot([0.15],[db_wp],'o',markersize=5)

ax4.plot([0.35],[db_ws],'o',markersize=5)

ax4.set_xlim(0,1)

ax4.set_ylim(-60,2)

ax4.set_title('Digital Butterworth Magnitude Response')

ax4.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax4.set_ylabel('Magnitude (dB)')

ax4.grid(True,linestyle=':',alpha=0.25)

ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.30,hspace=0.62)

# ============================================================
# DISPLAY FIGURE
# ============================================================

display(fig.canvas)